#### <span style="color:blue">Azul:</span> Aluno 1

#### <span style="color:red">Vermelho:</span> Gabriel

#### <span style="color:green">Verde:</span> Wendell

#### <span style="color:red">Gabriel</span>

#### Passos

1. **Pré-processamento**
- Agrupar dados em janelas de 30s (as classificações pelos médicos são feitas analisando os ultimos 30s, então podemos agrupar por blocos de 30s para simplificar a quantidade de dados)
- Reduzir quantidade de dados e padronizar análise

2. **Gráficos**
- Duração total do sono e fases
- Duração de cada fase
- Interrupções do sono
- Qualidade do sono
- Análise por vários dias / semanal
- Relação entre atividade do EEG e fases

3. **Validação**
- Treinar com EDF classificado (T1)
- Gerar gráficos de saída
- Aplicar em EDF sem classificação (T2)
- Comparar resultados

4. **Importante**
- Não usar os mesmos EDFs em treino e teste

In [19]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mne
from scipy.signal import welch
from scipy.stats import skew, kurtosis
from scipy.integrate import trapezoid

In [20]:
#2.1 Carregamento dos dados
# pasta com os arquivos EDF originais
dados_dir = Path("../dados/edfs_originais")
# pasta onde o CSV final sera salvo
saida_dir = Path("../dados/brutos")
dados_dir


WindowsPath('../dados/edfs_originais')

In [21]:
#3 Leitura e organiza??o
# extrai a chave comum usada para juntar PSG e hypnograma
def peganome(filename: str) -> str:
    base = Path(filename).name.split("-")[0]
    return base[:7]

# procura os arquivos e monta os pares correspondentes
def encontrapar(data_dir: Path, prefix="SC"):
    arquivospsg = sorted(data_dir.glob(f"{prefix}*-PSG.edf"))
    arquivoshyp = sorted(data_dir.glob(f"{prefix}*-Hypnogram.edf"))
    # dicionarios para buscar os arquivos pela chave do paciente/noite
    dicpsg = {peganome(arquivo.name): arquivo for arquivo in arquivospsg}
    dichyp = {peganome(arquivo.name): arquivo for arquivo in arquivoshyp}
    # mantem apenas as chaves que possuem os dois arquivos
    dicjunto = sorted(set(dicpsg) & set(dichyp))
    return [(k, dicpsg[k], dichyp[k]) for k in dicjunto]

# gera a lista de pares validos
pares = encontrapar(dados_dir, prefix="SC")
len(pares), pares[:5]


(1,
 [('SC4001E',
   WindowsPath('../dados/edfs_originais/SC4001E0-PSG.edf'),
   WindowsPath('../dados/edfs_originais/SC4001EC-Hypnogram.edf'))])

In [22]:
# Escolher quantidade de noites pra analisar
totalpares = 10
# seleciona so os primeiros pares para teste
selecpares = pares[:totalpares]
selecpares


[('SC4001E',
  WindowsPath('../dados/edfs_originais/SC4001E0-PSG.edf'),
  WindowsPath('../dados/edfs_originais/SC4001EC-Hypnogram.edf'))]

In [23]:
# Conferir se par foi carregado corretamente
chavepar, caminhopsg, caminhohyp = selecpares[0]

# le o sinal bruto e as anotacoes do hypnograma
bruto = mne.io.read_raw_edf(caminhopsg, preload=False, infer_types=True, verbose="ERROR")
anot = mne.read_annotations(caminhohyp)

print(f"Chave do par: {chavepar}")
print(f"PSG: {caminhopsg}")
print(f"Hypnograma: {caminhohyp}")
print(f"Canais: {bruto.ch_names}")
print(f"Frequ?ncia de amostragem: {bruto.info['sfreq']} Hz")
print("\nPrimeiras anota??es:")
# mostra um resumo inicial para validar as classes e os tempos
print(pd.DataFrame({
    "In?cio(s)": anot.onset[:10],
    "Dura??o(s)": anot.duration[:10],
    "Classifica??o": anot.description[:10]
}))


Chave do par: SC4001E
PSG: ..\dados\edfs_originais\SC4001E0-PSG.edf
Hypnograma: ..\dados\edfs_originais\SC4001EC-Hypnogram.edf
Canais: ['Fpz-Cz', 'Pz-Oz', 'horizontal', 'oro-nasal', 'submental', 'rectal', 'Event marker']
Frequ?ncia de amostragem: 100.0 Hz

Primeiras anota??es:
   In?cio(s)  Dura??o(s)  Classifica??o
0        0.0     30630.0  Sleep stage W
1    30630.0       120.0  Sleep stage 1
2    30750.0       390.0  Sleep stage 2
3    31140.0        30.0  Sleep stage 3
4    31170.0        30.0  Sleep stage 2
5    31200.0       150.0  Sleep stage 3
6    31350.0        30.0  Sleep stage 4
7    31380.0        60.0  Sleep stage 3
8    31440.0        60.0  Sleep stage 4
9    31500.0        30.0  Sleep stage 3


In [24]:
#4 Juntar em blocos de 30s, juntar estagio 3 e 4, selecionar canal principal, juntar com anotacoes, remover longas partes do acordado, extrair dados
# tamanho padrao de cada epoca em segundos
SEGUNDOS_EPOCA = 30.0
# faixa do filtro passa-banda aplicada ao EEG
FREQ_CORTE_BAIXA = 0.5
FREQ_CORTE_ALTA = 30.0
# canal principal usado na analise
CANAL_EEG = "Fpz-Cz"
# pares e pasta final reaproveitados nas proximas celulas
pares_sel = selecpares
PASTA_SAIDA = saida_dir
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

# mapeia os r?tulos do hypnograma para ids num?ricos
AnotacaoParaId = {
    "Sleep stage W": 1,
    "Sleep stage 1": 2,
    "Sleep stage 2": 3,
    "Sleep stage 3": 4, # juntar estagio 3 e 4
    "Sleep stage 4": 4, # juntar estagio 3 e 4
    "Sleep stage R": 5,
}

# traduz os ids para os nomes das classes finais
IdPraClassificacao = {
    1: "Acordado",
    2: "N1",
    3: "N2",
    4: "N3",
    5: "REM"
}

# valida se o canal desejado existe no arquivo lido
def escolher_canal(bruto, escolha="Fpz-Cz"):
    if escolha in bruto.ch_names:
        return escolha
    else:
        raise ValueError(f"Canal {escolha} n?o encontrado. Canais dispon?veis: {bruto.ch_names}")
    
# calcula a pot?ncia de uma faixa de frequ?ncia usando Welch
def calcular_potencia_banda(x, fs, fmin, fmax):
    amostras_janela = min(len(x), int(4 * fs))
    frequencias, densidade_potencia = welch(x, fs=fs, nperseg=amostras_janela)
    mascara_banda = (frequencias >= fmin) & (frequencias <= fmax)
    if not np.any(mascara_banda):
        return 0.0
    return trapezoid(densidade_potencia[mascara_banda], frequencias[mascara_banda])
    
# extrai estat?sticas simples e pot?ncias por banda de uma ?poca
def extrair_dados_epocas(x, fs):
    delta = calcular_potencia_banda(x, fs=fs, fmin=0.5, fmax=4)
    theta = calcular_potencia_banda(x, fs=fs, fmin=4, fmax=8)
    alpha = calcular_potencia_banda(x, fs=fs, fmin=8, fmax=13)
    beta = calcular_potencia_banda(x, fs=fs, fmin=13, fmax=30)
    total = calcular_potencia_banda(x, fs=fs, fmin=0.5, fmax=30)
    eps = 1e-12
    return {
        "media": np.mean(x),
        "desvio_padrao": np.std(x, ddof=1),
        "variancia": np.var(x, ddof=1),
        "minimo": np.min(x),
        "maximo": np.max(x),
        "pico_a_pico": np.ptp(x),
        "valor_rms": np.sqrt(np.mean(x**2)),
        "assimetria": skew(x, bias=False),
        "curtose_excesso": kurtosis(x, fisher=True, bias=False),
        "potencia_delta": delta,
        "potencia_theta": theta,
        "potencia_alpha": alpha,
        "potencia_beta": beta,
        "potencia_total": total,
        "relativo_delta": delta / (total + eps),
        "relativo_theta": theta / (total + eps),
        "relativo_alpha": alpha / (total + eps),
        "relativo_beta": beta / (total + eps),
        "razao_delta_theta": delta / (theta + eps),
        "razao_delta_alpha": delta / (alpha + eps),
    }


Proxima etapa
- Encapsular a logica em processar_par
- Gerar as epocas e extrair as features
- Concatenar os pares processados e salvar em CSV


#### <span style="color:green">Wendell




In [25]:
# encontra o intervalo util do sono com margem antes e depois
def lim_sono(anot, margem_segundos=1800):
    # classes consideradas como sono real
    estagios_sono = {
        "Sleep stage 1",
        "Sleep stage 2",
        "Sleep stage 3",
        "Sleep stage 4",
        "Sleep stage R",
    }
    ind_sono = np.where(np.isin(anot.description, list(estagios_sono)))[0]

    # se nao houver sono anotado, mantem a janela inteira
    if len(ind_sono) == 0:
        return 0, anot.onset[-1] + anot.duration[-1]

    primeiro_ind = ind_sono[0]
    ultimo_ind = ind_sono[-1]

    ini_recorte = max(0, anot.onset[primeiro_ind] - margem_segundos)
    fim_recorte = anot.onset[ultimo_ind] + anot.duration[ultimo_ind] + margem_segundos

    return ini_recorte, fim_recorte


In [26]:
# recorta as anotacoes para ficar so com a janela util do sono
def recortar_anotacoes_sono(anot, margem_segundos=1800):
    ini_recorte, fim_recorte = lim_sono(anot, margem_segundos=margem_segundos)
    anot_recortada = anot.copy()
    anot_recortada.crop(tmin=ini_recorte, tmax=fim_recorte)
    return anot_recortada

# processa um par PSG + hypnograma e devolve um dataframe de epocas
def processar_par(caminhopsg, caminhohyp, chavepar):
    # carrega o sinal e as anotacoes do par atual
    bruto = mne.io.read_raw_edf(caminhopsg, preload=True, infer_types=True, verbose="ERROR")
    anot = mne.read_annotations(caminhohyp)

    # limita a analise a janela util do sono
    anot = recortar_anotacoes_sono(anot, margem_segundos=1800)
    bruto.set_annotations(anot, emit_warning=False)

    # escolhe o canal principal e aplica o filtro
    canal_alvo = escolher_canal(bruto, CANAL_EEG)
    bruto.pick([canal_alvo])
    bruto.filter(FREQ_CORTE_BAIXA, FREQ_CORTE_ALTA, fir_design="firwin", verbose="ERROR")

    fs = float(bruto.info["sfreq"])

    # transforma as anotacoes em eventos de 30 segundos
    eventos, _ = mne.events_from_annotations(
        bruto,
        event_id=AnotacaoParaId,
        chunk_duration=SEGUNDOS_EPOCA,
        verbose="ERROR",
    )

    # gera as epocas alinhadas aos eventos encontrados
    epocas = mne.Epochs(
        bruto,
        eventos,
        event_id=None,
        tmin=0.0,
        tmax=SEGUNDOS_EPOCA - 1.0 / fs,
        baseline=None,
        preload=True,
        verbose="ERROR",
    )

    dados = epocas.get_data(copy=False)
    codigos_estagios = epocas.events[:, 2]
    amostras_eventos = epocas.events[:, 0]

    linhas = []
    for i in range(len(epocas)):
        x = dados[i, 0, :]
        # extrai as features da ?poca atual
        feat = extrair_dados_epocas(x, fs=fs)

        # monta uma linha com metadados e features da ?poca
        linha = {
            "chavepar": chavepar,
            "arquivo_psg": caminhopsg.name,
            "arquivo_hyp": caminhohyp.name,
            "canal": canal_alvo,
            "frequencia_amostragem_hz": fs,
            "indice_epoca": i,
            "inicio_epoca_s": amostras_eventos[i] / fs,
            "id_estagio": int(codigos_estagios[i]),
            "estagio": IdPraClassificacao.get(int(codigos_estagios[i]), "OUTRO"),
        }
        linha.update(feat)
        linhas.append(linha)

    return pd.DataFrame(linhas)

# exemplo com o primeiro par selecionado
df_exemplo = processar_par(caminhopsg, caminhohyp, chavepar)
print(f"{chavepar}: {len(df_exemplo)} epocas")
df_exemplo.head()


SC4001E: 841 epocas


,chavepar,arquivo_psg,arquivo_hyp,canal,frequencia_amostragem_hz,indice_epoca,inicio_epoca_s,id_estagio,estagio,media,...,potencia_theta,potencia_alpha,potencia_beta,potencia_total,relativo_delta,relativo_theta,relativo_alpha,relativo_beta,razao_delta_theta,razao_delta_alpha
0,SC4001E,SC4001E0-PSG.edf,SC4001EC-Hypnogram.edf,Fpz-Cz,100.0,0,28830.0,1,Acordado,-1.665176e-07,...,3.322136e-11,5.745580e-12,1.047336e-11,4.010510e-10,0.874543,0.082630,0.014291,0.026050,10.274598,52.124612
1,SC4001E,SC4001E0-PSG.edf,SC4001EC-Hypnogram.edf,Fpz-Cz,100.0,1,28860.0,1,Acordado,2.558510e-07,...,4.281629e-11,4.770206e-12,1.034115e-11,5.538721e-10,0.893800,0.077164,0.008597,0.018637,11.318721,85.949175
2,SC4001E,SC4001E0-PSG.edf,SC4001EC-Hypnogram.edf,Fpz-Cz,100.0,2,28890.0,1,Acordado,-1.937950e-07,...,6.198571e-11,5.257595e-12,8.410971e-12,8.054582e-10,0.904949,0.076862,0.006519,0.010430,11.586817,116.626897
3,SC4001E,SC4001E0-PSG.edf,SC4001EC-Hypnogram.edf,Fpz-Cz,100.0,3,28920.0,1,Acordado,-5.086672e-07,...,4.731464e-11,6.219261e-12,1.680778e-11,6.127919e-10,0.883769,0.077086,0.010133,0.027384,11.227449,75.139296
4,SC4001E,SC4001E0-PSG.edf,SC4001EC-Hypnogram.edf,Fpz-Cz,100.0,4,28950.0,1,Acordado,1.960303e-07,...,4.000793e-11,5.459383e-12,1.011999e-11,4.849642e-10,0.883557,0.082327,0.011234,0.020825,10.470582,66.473367


In [27]:
import traceback

dfs = []

# processa todos os pares selecionados e acumula os dataframes
for chavepar, caminhopsg, caminhohyp in pares_sel:
    print(f"\nProcessando {chavepar}")
    print("PSG:", caminhopsg)
    print("HYP:", caminhohyp)

    try:
        df_i = processar_par(caminhopsg, caminhohyp, chavepar)
        dfs.append(df_i)
        print(f"{chavepar}: {len(df_i)} epocas")
    except Exception as e:
        print(f"Erro em {chavepar}: {type(e).__name__}: {e}")
        traceback.print_exc()

if len(dfs) == 0:
    raise RuntimeError(
        "Nenhum arquivo foi processado com sucesso. "
        "Veja os erros impressos acima."
    )

# junta tudo em um unico dataframe e salva em CSV
df = pd.concat(dfs, ignore_index=True)
df.to_csv(PASTA_SAIDA / "dados_epocas.csv", index=False)

df.head()



Processando SC4001E
PSG: ..\dados\edfs_originais\SC4001E0-PSG.edf
HYP: ..\dados\edfs_originais\SC4001EC-Hypnogram.edf
SC4001E: 841 epocas


,chavepar,arquivo_psg,arquivo_hyp,canal,frequencia_amostragem_hz,indice_epoca,inicio_epoca_s,id_estagio,estagio,media,...,potencia_theta,potencia_alpha,potencia_beta,potencia_total,relativo_delta,relativo_theta,relativo_alpha,relativo_beta,razao_delta_theta,razao_delta_alpha
0,SC4001E,SC4001E0-PSG.edf,SC4001EC-Hypnogram.edf,Fpz-Cz,100.0,0,28830.0,1,Acordado,-1.665176e-07,...,3.322136e-11,5.745580e-12,1.047336e-11,4.010510e-10,0.874543,0.082630,0.014291,0.026050,10.274598,52.124612
1,SC4001E,SC4001E0-PSG.edf,SC4001EC-Hypnogram.edf,Fpz-Cz,100.0,1,28860.0,1,Acordado,2.558510e-07,...,4.281629e-11,4.770206e-12,1.034115e-11,5.538721e-10,0.893800,0.077164,0.008597,0.018637,11.318721,85.949175
2,SC4001E,SC4001E0-PSG.edf,SC4001EC-Hypnogram.edf,Fpz-Cz,100.0,2,28890.0,1,Acordado,-1.937950e-07,...,6.198571e-11,5.257595e-12,8.410971e-12,8.054582e-10,0.904949,0.076862,0.006519,0.010430,11.586817,116.626897
3,SC4001E,SC4001E0-PSG.edf,SC4001EC-Hypnogram.edf,Fpz-Cz,100.0,3,28920.0,1,Acordado,-5.086672e-07,...,4.731464e-11,6.219261e-12,1.680778e-11,6.127919e-10,0.883769,0.077086,0.010133,0.027384,11.227449,75.139296
4,SC4001E,SC4001E0-PSG.edf,SC4001EC-Hypnogram.edf,Fpz-Cz,100.0,4,28950.0,1,Acordado,1.960303e-07,...,4.000793e-11,5.459383e-12,1.011999e-11,4.849642e-10,0.883557,0.082327,0.011234,0.020825,10.470582,66.473367
